# qwen21-nsfw-pose · Colab 快速版（一键）

> 先在「代码执行程序 → 更改运行时类型」选 **T4 GPU**。
>
> 三个 cell 依次运行：① 挂载 Drive 并克隆仓库 → ② 运行 `scripts/colab_quickstart.py`（装依赖、取底模/LoRA、启动 ComfyUI、出图）→ ③ 显示结果。
>
> 所有步骤可重复运行：已有的文件会跳过，ComfyUI 已在运行就直接复用，所以再出一张图只需重跑 ② 和 ③。
>
> 首次运行需要 LoRA：把文件放进 Drive 的 `qwen21_pose/loras/`，或在 ② 里加 `--lora-url ... --civitai-token ...`。
> 底模默认缓存到 Drive（约 16GB，需账号空间足够；不想缓存加 `--no-cache-base`）。
>
> ⚠️ 未在所有 T4 环境实测；若 OOM 或黑图，参见仓库 `install/colab.md`。

In [ ]:
# ===== ① 挂载 Drive + 克隆仓库 =====
REPO_URL = "https://github.com/ldybyz/qwen21-nsfw-pose-skill.git"
BRANCH = "feat/colab-t4-guide"  # 脚本所在分支；合并进 main 后可改回 main
POSELAB = "/content/qwen21-nsfw-pose-skill"

from google.colab import drive
drive.mount("/content/drive")

!test -d {POSELAB} || git clone -b {BRANCH} {REPO_URL} {POSELAB}
# 目录已存在时也强制切到目标分支并更新（避免停留在旧分支）
!cd {POSELAB} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}

In [ ]:
# ===== ② 一键：装依赖 / 取模型 / 启动 ComfyUI / 出图 =====
# 常用参数（按需增删）：
#   --lora-url "https://civitai.com/api/download/models/<版本ID>" --civitai-token "<token>"   LoRA 不在 Drive 时下载
#   --width 1024 --height 1024 --steps 20 --seed 80964                                      出图参数（降配需如实标注）
#   --prompt "..." 或 --prompt-file prompts/xxx.txt                                         自定义提示词
#   --no-cache-base                                                                          不把底模缓存到 Drive
#   --no-drive                                                                               完全不用 Drive
!python {POSELAB}/scripts/colab_quickstart.py --steps 20 --prompt-file /content/drive/MyDrive/qwen21_pose/prompts/04_topdown_silk.txt

In [ ]:
# ===== ③ 显示最新一张图 =====
import glob, os
from IPython.display import Image, display

imgs = sorted(glob.glob(f"{POSELAB}/outputs/colab_*.png"), key=os.path.getmtime)
if imgs:
    print(imgs[-1])
    display(Image(imgs[-1]))
else:
    print("还没有输出图，先运行 ②")

In [ ]:
# ===== ④ （可选）改已有的图：换姿势 / 改场景，保留人物 =====
# 先把原图上传到 Drive 的 qwen21_pose/inputs/，再改下面两项；跑完用 ③ 查看结果。
# 默认不带 NSFW LoRA、cfg 3、20 步；指令用英文、SFW 措辞最稳（详见 install/colab_usage.md）。
INPUT_IMAGE = "/content/drive/MyDrive/qwen21_pose/inputs/my_image.png"
INSTRUCTION = "change her pose to kneeling on one knee, hands on hips, keep face hair and outfit"

!python {POSELAB}/scripts/colab_quickstart.py --mode edit --input "{INPUT_IMAGE}" --prompt "{INSTRUCTION}" --skip-doctor